# Example 1

Dynamical system is
$$ \dot x = \begin{bmatrix} -2x_1 \\ -3(x_2 - x_1^2) \end{bmatrix} $$

In [1]:
import numpy as np
from scipy.linalg import eig
import matplotlib.pyplot as plt
import cvxpy as cp
import time

In [2]:
#Define Jacobian matrix E= dF(0)/dz at origin
E = np.array([[-2,0],[0,-3]])
# compute eigvalues and left eigenvectors of E
e,evecL,evecR = eig(E,left=True)
# eigenvalues
eval1 = e[0].real
eval2 = e[1].real
# eigenvectors
w1 = evecL[:,0]
w2 = evecL[:,1]

In [3]:
# Define (z_1,z_2) grid
npoints = 30
x_grid = np.linspace(-5, 5, npoints)
y_grid = np.linspace(-5, 5, npoints)

X,Y = np.meshgrid(x_grid,y_grid)
XY = np.zeros((npoints**2,2))

k = 0
for i in range(npoints):
    for j in range(npoints):
        XY[k,:] = np.array([X[i,j],Y[i,j]])
        k = k+1


# Define test grids
#n_test = 60
#x_grid_test = np.linspace(-5, 5, n_test)
#y_grid_test = np.linspace(-5, 5, n_test)

#X_test,Y_test = np.meshgrid(x_grid_test,y_grid_test)
#XY_test = np.zeros((n_test**2,2))

#k = 0
#for i in range(n_test):
#    for j in range(n_test):
#        XY_test[k,:] = np.array([X_test[i,j],Y_test[i,j]])
#        k = k+1

In [4]:
#Define functions F and G
def F(x):
    value1 = -2 * x[:,0]
    value2 = -3 * (x[:,1] - x[:,0]**2)
    return np.array([value1, value2]).T

def G(x):
    value = np.zeros((len(x),2))
    for i in range(len(x)):
        value[i,:] = np.dot(E, x[i,:])
    value = F(x)-value
    return value

In [5]:
# define eigenfunctions
def phi_1(X):
    return X[:,0]

def phi_2(X):
    return X[:,1] + 3 * X[:,0] ** 2

In [6]:
# copute F and G at mesh grids
F_val = F(XY)
G_val = G(XY)

## Kernel

In [7]:
## Define 2D Gaussian kernel and derivatives
def kernel(s1,t1,s2,t2,sigma1,sigma2):
    return np.exp(-((s1-t1)**2/(2*sigma1**2))-((s2-t2)**2/(2*sigma2**2)))

def kernel_dx1(s1,t1,s2,t2,sigma1,sigma2):
    return -(s1-t1)*kernel(s1,t1,s2,t2,sigma1,sigma2)/(sigma1**2)

def kernel_dy1(s1,t1,s2,t2,sigma1,sigma2):
    return -kernel_dx1(s1,t1,s2,t2,sigma1,sigma2)
  

def kernel_dx2(s1,t1,s2,t2,sigma1,sigma2):
    return -(s2-t2)*kernel(s1,t1,s2,t2,sigma1,sigma2)/(sigma2**2)

def kernel_dy2(s1,t1,s2,t2,sigma1,sigma2):
    return -kernel_dx2(s1,t1,s2,t2,sigma1,sigma2)
  

def kernel_dx1dy1(s1,t1,s2,t2,sigma1,sigma2):
    return (sigma1**2-(s1-t1)**2)*kernel(s1,t1,s2,t2,sigma1,sigma2)/(sigma1**4)

def kernel_dx1dy2(s1,t1,s2,t2,sigma1,sigma2):
    return (t2-s2)*(s1-t1)*kernel(s1,t1,s2,t2,sigma1,sigma2)/(sigma1**2*sigma2**2)

def kernel_dx2dy1(s1,t1,s2,t2,sigma1,sigma2):
    return kernel_dx1dy2(s1,t1,s2,t2,sigma1,sigma2)
   

def kernel_dx2dy2(s1,t1,s2,t2,sigma1,sigma2):
    return (sigma2**2-(s2-t2)**2)*kernel(s1,t1,s2,t2,sigma1,sigma2)/(sigma2**4)

In [8]:
#Generate matrix K(Z,Z) , where Z=(z_1,z_2): K(Z,Z)_{ij}=K(Z_i,Z_j)= K((z_1,z_2)_i,(z_1,z_2)_j)

def K_Anisotropic(X,Y,sigma1=1,sigma2=1,reg=False,nugget=10**-3):
    size=len(X[:,0])
    X0=np.transpose(np.tile(X[:,0],(size,1)))
    X1=np.transpose(np.tile(X[:,1],(size,1)))
    Y0=np.transpose(np.tile(Y[:,0],(size,1)))
    Y1=np.transpose(np.tile(Y[:,1],(size,1)))

    val = np.vectorize(lambda s1, t1, s2, t2: kernel(s1, t1,s2,t2,sigma1,sigma2))(X0.flatten(),np.transpose(Y0).flatten(),X1.flatten(),np.transpose(Y1).flatten())

    K_matrix=np.reshape(val,(size,size))
    if reg==True:
        K_matrix+=+nugget*np.eye(size)
    return K_matrix

In [9]:
#Derivatives of K(Z,Z) with respects to x_1,x_2,y_1,y_2

def K_Anisotropic_dx1(X,Y,sigma1=1,sigma2=1,reg=False,nugget=10**-3):
    size=len(X[:,0])
    X0=np.transpose(np.tile(X[:,0],(size,1)))
    X1=np.transpose(np.tile(X[:,1],(size,1)))
    Y0=np.transpose(np.tile(Y[:,0],(size,1)))
    Y1=np.transpose(np.tile(Y[:,1],(size,1)))

    val = np.vectorize(lambda s1, t1, s2, t2: kernel_dx1(s1, t1,s2,t2,sigma1,sigma2))(X0.flatten(),np.transpose(Y0).flatten(),X1.flatten(),np.transpose(Y1).flatten())
    K_matrix=np.reshape(val,(size,size))
    if reg==True:
        K_matrix+=+nugget*np.eye(size)
    return K_matrix

def K_Anisotropic_dy1(X,Y,sigma1=1,sigma2=1,reg=False,nugget=10**-3):
    size=len(X[:,0])
    X0=np.transpose(np.tile(X[:,0],(size,1)))
    X1=np.transpose(np.tile(X[:,1],(size,1)))
    Y0=np.transpose(np.tile(Y[:,0],(size,1)))
    Y1=np.transpose(np.tile(Y[:,1],(size,1)))

    val = np.vectorize(lambda s1, t1, s2, t2: kernel_dy1(s1, t1,s2,t2,sigma1,sigma2))(X0.flatten(),np.transpose(Y0).flatten(),X1.flatten(),np.transpose(Y1).flatten())
    K_matrix=np.reshape(val,(size,size))
    if reg==True:
        K_matrix+=+nugget*np.eye(size)
    return K_matrix

def K_Anisotropic_dx2(X,Y,sigma1=1,sigma2=1,reg=False,nugget=10**-3):
    size=len(X[:,0])
    X0=np.transpose(np.tile(X[:,0],(size,1)))
    X1=np.transpose(np.tile(X[:,1],(size,1)))
    Y0=np.transpose(np.tile(Y[:,0],(size,1)))
    Y1=np.transpose(np.tile(Y[:,1],(size,1)))

    val = np.vectorize(lambda s1, t1, s2, t2: kernel_dx2(s1, t1,s2,t2,sigma1,sigma2))(X0.flatten(),np.transpose(Y0).flatten(),X1.flatten(),np.transpose(Y1).flatten())
    K_matrix=np.reshape(val,(size,size))
    if reg==True:
        K_matrix+=+nugget*np.eye(size)
    return K_matrix

def K_Anisotropic_dy2(X,Y,sigma1=1,sigma2=1,reg=False,nugget=10**-3):
    size=len(X[:,0])
    X0=np.transpose(np.tile(X[:,0],(size,1)))
    X1=np.transpose(np.tile(X[:,1],(size,1)))
    Y0=np.transpose(np.tile(Y[:,0],(size,1)))
    Y1=np.transpose(np.tile(Y[:,1],(size,1)))

    val = np.vectorize(lambda s1, t1, s2, t2: kernel_dy2(s1, t1,s2,t2,sigma1,sigma2))(X0.flatten(),np.transpose(Y0).flatten(),X1.flatten(),np.transpose(Y1).flatten())
    K_matrix=np.reshape(val,(size,size))
    if reg==True:
        K_matrix+=+nugget*np.eye(size)
    return K_matrix

In [10]:
def K_Anisotropic_dx1dy1(X,Y,sigma1=1,sigma2=1,reg=False,nugget=10**-3):
    size=len(X[:,0])
    X0=np.transpose(np.tile(X[:,0],(size,1)))
    X1=np.transpose(np.tile(X[:,1],(size,1)))
    Y0=np.transpose(np.tile(Y[:,0],(size,1)))
    Y1=np.transpose(np.tile(Y[:,1],(size,1)))

    val = np.vectorize(lambda s1, t1, s2, t2: kernel_dx1dy1(s1, t1,s2,t2,sigma1,sigma2))(X0.flatten(),np.transpose(Y0).flatten(),X1.flatten(),np.transpose(Y1).flatten())
    K_matrix=np.reshape(val,(size,size))
    if reg==True:
        K_matrix+=+nugget*np.eye(size)
    return K_matrix

def K_Anisotropic_dx1dy2(X,Y,sigma1=1,sigma2=1,reg=False,nugget=10**-3):
    size=len(X[:,0])
    X0=np.transpose(np.tile(X[:,0],(size,1)))
    X1=np.transpose(np.tile(X[:,1],(size,1)))
    Y0=np.transpose(np.tile(Y[:,0],(size,1)))
    Y1=np.transpose(np.tile(Y[:,1],(size,1)))

    val = np.vectorize(lambda s1, t1, s2, t2: kernel_dx1dy2(s1, t1,s2,t2,sigma1,sigma2))(X0.flatten(),np.transpose(Y0).flatten(),X1.flatten(),np.transpose(Y1).flatten())
    K_matrix=np.reshape(val,(size,size))
    if reg==True:
        K_matrix+=+nugget*np.eye(size)
    return K_matrix

def K_Anisotropic_dx2dy1(X,Y,sigma1=1,sigma2=1,reg=False,nugget=10**-3):
    size=len(X[:,0])
    X0=np.transpose(np.tile(X[:,0],(size,1)))
    X1=np.transpose(np.tile(X[:,1],(size,1)))
    Y0=np.transpose(np.tile(Y[:,0],(size,1)))
    Y1=np.transpose(np.tile(Y[:,1],(size,1)))

    val = np.vectorize(lambda s1, t1, s2, t2: kernel_dx2dy1(s1, t1,s2,t2,sigma1,sigma2))(X0.flatten(),np.transpose(Y0).flatten(),X1.flatten(),np.transpose(Y1).flatten())
    K_matrix=np.reshape(val,(size,size))
    if reg==True:
        K_matrix+=+nugget*np.eye(size)
    return K_matrix

def K_Anisotropic_dx2dy2(X,Y,sigma1=1,sigma2=1,reg=False,nugget=10**-3):
    size=len(X[:,0])
    X0=np.transpose(np.tile(X[:,0],(size,1)))
    X1=np.transpose(np.tile(X[:,1],(size,1)))
    Y0=np.transpose(np.tile(Y[:,0],(size,1)))
    Y1=np.transpose(np.tile(Y[:,1],(size,1)))

    val = np.vectorize(lambda s1, t1, s2, t2: kernel_dx2dy2(s1, t1,s2,t2,sigma1,sigma2))(X0.flatten(),np.transpose(Y0).flatten(),X1.flatten(),np.transpose(Y1).flatten())
    K_matrix=np.reshape(val,(size,size))
    if reg==True:
        K_matrix+=+nugget*np.eye(size)
    return K_matrix

In [11]:
def K_vector(X_train,X_test):
    size=len(X_train[:,0])+3
    size2=len(X_test[:,0])
    K_vec = np.zeros((size2,size))

    K_vec[:,0] = kernel(X_test[:,0],0,X_test[:,1],0,sigma1,sigma2)
    K_vec[:,1] = kernel_dy1(X_test[:,0],0,X_test[:,1],0,sigma1,sigma2)
    K_vec[:,2] = kernel_dy2(X_test[:,0],0,X_test[:,1],0,sigma1,sigma2)
    K_vec[:,3:] = K_Anisotropic_dy1(X_test,X_train,sigma1,sigma2)*F1j+K_Anisotropic_dy2(X_test,X_train,sigma1,sigma2)*F2j-eval1*K_Anisotropic(X_test,X_train,sigma1,sigma2)

    return K_vec

#Representer K(z,phi)(K(phi,phi)+lambda*I)^-1 Y
def kernel_regression(X_train,X_test,Y_train,K_Matrix,nugget=10**-3):
    t_matrix = K_vector(X_train,X_test)
    regressor = np.matmul(t_matrix,np.linalg.inv(K_Matrix+nugget*np.eye(len(K_Matrix[:,0])))@Y_train)
    return regressor

In [12]:
#Construct matrices for F

F11 = np.outer(F_val[:,0],F_val[:,0])
F12 = np.outer(F_val[:,0],F_val[:,1])
F21 = np.outer(F_val[:,1],F_val[:,0])
F22 = np.outer(F_val[:,1],F_val[:,1])

F1i = F_val[:,0].reshape((npoints**2,1))
F1j = F_val[:,0]
F2i = F_val[:,1].reshape((npoints**2,1))
F2j = F_val[:,1]

In [13]:
#Construct matrix K(phi,phi)
start1 = time.time()
K = np.zeros((npoints**2+3,npoints**2+3))

sigma1 = 2
sigma2 = 2

Kx1y1 = K_Anisotropic_dx1dy1(XY,XY,sigma1,sigma2)
Kx1y2 = K_Anisotropic_dx1dy2(XY,XY,sigma1,sigma2)
Kx2y1 = K_Anisotropic_dx2dy1(XY,XY,sigma1,sigma2)
Kx2y2 = K_Anisotropic_dx2dy2(XY,XY,sigma1,sigma2)

Kdy1 = kernel_dy1(0,XY[:,0],0,XY[:,1],sigma1,sigma2)
Kdy2 = kernel_dy2(0,XY[:,0],0,XY[:,1],sigma1,sigma2)
Kdx1y1 = kernel_dx1dy1(0,XY[:,0],0,XY[:,1],sigma1,sigma2)
Kdx1y2 = kernel_dx1dy2(0,XY[:,0],0,XY[:,1],sigma1,sigma2)
Kdx2y1 = kernel_dx2dy1(0,XY[:,0],0,XY[:,1],sigma1,sigma2)
Kdx2y2 = kernel_dx2dy2(0,XY[:,0],0,XY[:,1],sigma1,sigma2)


Kx1 = K_Anisotropic_dx1(XY,XY,sigma1,sigma2)
Kx2 = K_Anisotropic_dx2(XY,XY,sigma1,sigma2)
Ky1 = K_Anisotropic_dy1(XY,XY,sigma1,sigma2)
Ky2 = K_Anisotropic_dy2(XY,XY,sigma1,sigma2)

# h(0) = 0
K[0,0] = kernel(0,0,0,0,sigma1,sigma2)
# \nabla h(0) = 0
K[0,1] = kernel_dy1(0,0,0,0,sigma1,sigma2)
K[1,0] = K[0,1]
K[1,1] = kernel_dx1dy1(0,0,0,0,sigma1,sigma2)
K[2,2] = kernel_dx2dy2(0,0,0,0,sigma1,sigma2)
K[0,2] = kernel_dy2(0,0,0,0,sigma1,sigma2)
K[2,0] = K[0,2]
K[2,1] = kernel_dx1dy2(0,0,0,0,sigma1,sigma2)
K[1,2] = K[2,1]
K[0,3:] = F_val[:,0]*Kdy1+F_val[:,1]*Kdy2-eval1*kernel(XY[:,0],0,XY[:,1],0,sigma1,sigma2)
K[3:,0] = K[0,3:]
K[1,3:] = F_val[:,0]*Kdx1y1+F_val[:,1]*Kdx1y2-eval1*kernel_dy1(XY[:,0],0,XY[:,1],0,sigma1,sigma2)
K[3:,1] = K[1,3:]
K[2,3:] = F_val[:,0]*Kdx2y1+F_val[:,1]*Kdx2y2-eval1*kernel_dy2(XY[:,0],0,XY[:,1],0,sigma1,sigma2)
K[3:,2] = K[2,3:]
# linear PDE
K[3:,3:] = F11*Kx1y1+F12*Kx1y2+Kx2y1*F21+Kx2y2*F22-eval1*(Kx1*F1i+Kx2*F2i+Ky1*F1j+F2j*Ky2)+eval1**2*K_Anisotropic(XY,XY,sigma1,sigma2)

# right-hand side
y0 = np.zeros((3,1))
y1 = np.zeros((npoints**2,1))
for i in range(npoints**2):
    y1[i] = -G_val[i,:] @ w1
y = np.row_stack((y0,y1))

result = kernel_regression(XY,XY,y,K,nugget=1*10**-6)
end1 = time.time()
h_kernel = result.reshape((npoints,npoints))
phi1_kernel = h_kernel+w1[0]*X+w1[1]*Y
# compute true eigenfucntion
phi1 = phi_1(XY).reshape(npoints,npoints)

# compute averaged test error
# test error
test_error = np.mean( (phi1_kernel - phi1) ** 2 )
print(f'Average test error is {test_error:.2e}')

# relative test error
rel_error = np.sum((phi1_kernel - phi1) ** 2) / np.sum(phi1**2)
print(f'Relative test error is {rel_error:.2e}')

Average test error is 0.00e+00
Relative test error is 0.00e+00


In [24]:
#Construct matrix K(phi,phi)
start2 = time.time()
K = np.zeros((npoints**2+3,npoints**2+3))

sigma1 = 0.1
sigma2 = 0.1

Kx1y1 = K_Anisotropic_dx1dy1(XY,XY,sigma1,sigma2)
Kx1y2 = K_Anisotropic_dx1dy2(XY,XY,sigma1,sigma2)
Kx2y1 = K_Anisotropic_dx2dy1(XY,XY,sigma1,sigma2)
Kx2y2 = K_Anisotropic_dx2dy2(XY,XY,sigma1,sigma2)

Kdy1 = kernel_dy1(0,XY[:,0],0,XY[:,1],sigma1,sigma2)
Kdy2 = kernel_dy2(0,XY[:,0],0,XY[:,1],sigma1,sigma2)
Kdx1y1 = kernel_dx1dy1(0,XY[:,0],0,XY[:,1],sigma1,sigma2)
Kdx1y2 = kernel_dx1dy2(0,XY[:,0],0,XY[:,1],sigma1,sigma2)
Kdx2y1 = kernel_dx2dy1(0,XY[:,0],0,XY[:,1],sigma1,sigma2)
Kdx2y2 = kernel_dx2dy2(0,XY[:,0],0,XY[:,1],sigma1,sigma2)


Kx1 = K_Anisotropic_dx1(XY,XY,sigma1,sigma2)
Kx2 = K_Anisotropic_dx2(XY,XY,sigma1,sigma2)
Ky1 = K_Anisotropic_dy1(XY,XY,sigma1,sigma2)
Ky2 = K_Anisotropic_dy2(XY,XY,sigma1,sigma2)

# h(0) = 0
K[0,0] = kernel(0,0,0,0,sigma1,sigma2)
# \nabla h(0) = 0
K[0,1] = kernel_dy1(0,0,0,0,sigma1,sigma2)
K[1,0] = K[0,1]
K[1,1] = kernel_dx1dy1(0,0,0,0,sigma1,sigma2)
K[2,2] = kernel_dx2dy2(0,0,0,0,sigma1,sigma2)
K[0,2] = kernel_dy2(0,0,0,0,sigma1,sigma2)
K[2,0] = K[0,2]
K[2,1] = kernel_dx1dy2(0,0,0,0,sigma1,sigma2)
K[1,2] = K[2,1]
K[0,3:] = F_val[:,0]*Kdy1+F_val[:,1]*Kdy2-eval2*kernel(XY[:,0],0,XY[:,1],0,sigma1,sigma2)
K[3:,0] = K[0,3:]
K[1,3:] = F_val[:,0]*Kdx1y1+F_val[:,1]*Kdx1y2-eval2*kernel_dy1(XY[:,0],0,XY[:,1],0,sigma1,sigma2)
K[3:,1] = K[1,3:]
K[2,3:] = F_val[:,0]*Kdx2y1+F_val[:,1]*Kdx2y2-eval2*kernel_dy2(XY[:,0],0,XY[:,1],0,sigma1,sigma2)
K[3:,2] = K[2,3:]
# linear PDE
K[3:,3:] = F11*Kx1y1+F12*Kx1y2+Kx2y1*F21+Kx2y2*F22-eval2*(Kx1*F1i+Kx2*F2i+Ky1*F1j+F2j*Ky2)+eval2**2*K_Anisotropic(XY,XY,sigma1,sigma2)

# right-hand side
y0 = np.zeros((3,1))
y1 = np.zeros((npoints**2,1))
for i in range(npoints**2):
    y1[i] = -G_val[i,:] @ w2
y = np.row_stack((y0,y1))

result = kernel_regression(XY,XY,y,K,nugget=1*10**-6)
end2 = time.time()
h_kernel = result.reshape((npoints,npoints))
phi2_kernel = h_kernel+w2[0]*X+w2[1]*Y
# compute true eigenfucntion
phi2 = phi_2(XY).reshape(npoints,npoints)

# compute averaged test error
# test error
test_error = np.mean( (phi2_kernel - phi2) ** 2 )
print(f'Average test error is {test_error:.2e}')

# relative test error
rel_error = np.sum((phi2_kernel - phi2) ** 2) / np.sum(phi2**2)
print(f'Relative test error is {rel_error:.2e}')

Average test error is 1.28e+03
Relative test error is 9.93e-01


In [25]:
# computation time is
print(f'{end1-start1 + end2-start2:.2f} seconds')

43.60 seconds
